In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ── Cell 1b · Setup: import + load probability files ──────────────────────────
# (This cell is inserted between Cell 1 [Drive mount] and Cell 2 [Data structure check].)
import sys
sys.path.insert(0, '/content/drive/MyDrive/0000_Kaan_CDS/src')

import importlib
for _m in ['plot_style', 'cds_config']:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

from cds_config import *          # PATHS, SCENARIOS, S2_CAL_COLS, CLASS_COLORS, setup_notebook ...
import pandas as pd
import numpy as np
from pathlib import Path

setup_notebook("Chat 1 — M1 Calibration")

# ── Populate the prob_data dictionary ─────────────────────────────────────────
# Key format:   stage{1,2}_{CBC_Only,CBC_BIO}_{oof,test}
# File format:  stage{1,2}_{cbc_only,cbc_bio}_{oof,test}_probs.parquet  (lowercase)
PROB_DIR = PATHS.probabilities

prob_data = {}
for stage in [1, 2]:
    for scn in SCENARIOS:                       # SCENARIOS = ['CBC_Only', 'CBC_BIO']
        for split in ['oof', 'test']:
            key   = f"stage{stage}_{scn}_{split}"
            fname = f"stage{stage}_{scn.lower()}_{split}_probs.parquet"
            fpath = PROB_DIR / fname
            prob_data[key] = pd.read_parquet(fpath)

print(f"\n✓ prob_data loaded — {len(prob_data)} files")
for k, df in prob_data.items():
    print(f"  {k:35s} → {df.shape[0]:>4d} rows, {df.shape[1]:>2d} cols")

## Cell 2 · Data Structure Check

In [ ]:
# ── Cell 2 · Data structure check ─────────────────────────────────────

# Stage 1 column structure
print("═" * 60)
print("STAGE 1 — Column structure (CBC_Only OOF)")
print("═" * 60)
df1 = prob_data["stage1_CBC_Only_oof"]
prob_cols_1 = [c for c in df1.columns if c.startswith("prob_")]
meta_cols_1 = ["target", "diagnosis", "pred", "correct", "confidence"]
print(f"  Prob columns : {prob_cols_1}")
print(f"  Meta columns : {[c for c in meta_cols_1 if c in df1.columns]}")
print(f"  Target distribution:\n{df1['target'].value_counts().to_string()}")
print(f"  Prob range   : [{df1[prob_cols_1[0]].min():.4f}, {df1[prob_cols_1[0]].max():.4f}]")

print()

# Stage 2 column structure
print("═" * 60)
print("STAGE 2 — Column structure (CBC_Only OOF)")
print("═" * 60)
df2 = prob_data["stage2_CBC_Only_oof"]
prob_cols_2 = [c for c in df2.columns if c.startswith("prob_")]
meta_cols_2 = ["target", "target_label", "diagnosis", "pred", "pred_label", "correct", "confidence"]
print(f"  Prob columns : {prob_cols_2}")
print(f"  Meta columns : {[c for c in meta_cols_2 if c in df2.columns]}")
print(f"  Target distribution:\n{df2['target_label'].value_counts().to_string()}")

# Probability sum check (Stage 2)
row_sums = df2[prob_cols_2].sum(axis=1)
print(f"\n  Prob row sum: min={row_sums.min():.4f}, max={row_sums.max():.4f}, mean={row_sums.mean():.4f}")

# OOF / test size consistency
print()
print("═" * 60)
print("SIZE SUMMARY")
print("═" * 60)
for key, df in prob_data.items():
    print(f"  {key:35s} → {df.shape[0]:>4d} rows, {df.shape[1]:>2d} cols")

## Cell 3 · Calibration Metrics

In [ ]:
# ── Cell 3 · Calibration metrics ───────────────────────────────────────
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
import joblib

# ── ECE & MCE computation ─────────────────────────────────────────────
def calc_ece_mce(y_true, y_prob, n_bins=10):
    """Expected & Maximum Calibration Error (equal-width bins)."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece, mce = 0.0, 0.0
    bin_details = []

    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob > lo) & (y_prob <= hi)
        if lo == 0:  # first bin: include the left edge
            mask = (y_prob >= lo) & (y_prob <= hi)
        n_bin = mask.sum()
        if n_bin == 0:
            continue
        avg_conf = y_prob[mask].mean()
        avg_acc  = y_true[mask].mean()
        gap = abs(avg_acc - avg_conf)
        ece += (n_bin / len(y_true)) * gap
        mce = max(mce, gap)
        bin_details.append({
            "bin_lo": lo, "bin_hi": hi,
            "n": n_bin, "avg_conf": avg_conf,
            "avg_acc": avg_acc, "gap": gap
        })
    return ece, mce, bin_details

# ── Binary calibrators (Stage 1) ─────────────────────────────────────
def fit_platt(y_true, y_prob):
    """Platt Scaling: logistic regression on logit(prob)."""
    lr = LogisticRegression(C=1e10, solver='lbfgs', max_iter=5000)
    logits = np.log(np.clip(y_prob, 1e-8, 1 - 1e-8) /
                    (1 - np.clip(y_prob, 1e-8, 1 - 1e-8)))
    lr.fit(logits.reshape(-1, 1), y_true)
    return lr

def transform_platt(lr, y_prob):
    logits = np.log(np.clip(y_prob, 1e-8, 1 - 1e-8) /
                    (1 - np.clip(y_prob, 1e-8, 1 - 1e-8)))
    return lr.predict_proba(logits.reshape(-1, 1))[:, 1]

def fit_isotonic(y_true, y_prob):
    """Isotonic Regression calibration."""
    iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip')
    iso.fit(y_prob, y_true)
    return iso

def transform_isotonic(iso, y_prob):
    return iso.predict(y_prob)

# ── Multiclass calibrators (Stage 2) ─────────────────────────────────
def fit_multiclass_isotonic(y_true_onehot, y_prob_matrix, class_names):
    """Independent isotonic regression per class (OvR approach)."""
    calibrators = {}
    for i, cls in enumerate(class_names):
        iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip')
        iso.fit(y_prob_matrix[:, i], y_true_onehot[:, i])
        calibrators[cls] = iso
    return calibrators

def transform_multiclass_isotonic(calibrators, y_prob_matrix, class_names):
    """Calibrate + normalize (row sum = 1)."""
    cal_probs = np.zeros_like(y_prob_matrix)
    for i, cls in enumerate(class_names):
        cal_probs[:, i] = calibrators[cls].predict(y_prob_matrix[:, i])
    # Normalize: make each row sum to 1
    row_sums = cal_probs.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)  # zero-division guard
    cal_probs = cal_probs / row_sums
    return cal_probs

# ── Summary metric computation ─────────────────────────────────────────
def binary_calibration_metrics(y_true, y_prob, n_bins=10, label=""):
    """All calibration metrics for binary classification."""
    brier = brier_score_loss(y_true, y_prob)
    ece, mce, bins = calc_ece_mce(y_true, y_prob, n_bins=n_bins)
    return {"label": label, "Brier": brier, "ECE": ece, "MCE": mce,
            "n_bins_used": len(bins), "bins": bins}

def multiclass_calibration_metrics(y_true_onehot, y_prob_matrix,
                                    class_names, n_bins=10, label=""):
    """Multiclass: per-class Brier + ECE, then macro average."""
    results = {"label": label}
    briers, eces, mces = [], [], []
    for i, cls in enumerate(class_names):
        b = brier_score_loss(y_true_onehot[:, i], y_prob_matrix[:, i])
        e, m, _ = calc_ece_mce(y_true_onehot[:, i], y_prob_matrix[:, i], n_bins)
        results[f"Brier_{cls}"] = b
        results[f"ECE_{cls}"] = e
        results[f"MCE_{cls}"] = m
        briers.append(b); eces.append(e); mces.append(m)
    results["Brier_macro"] = np.mean(briers)
    results["ECE_macro"]   = np.mean(eces)
    results["MCE_macro"]   = np.mean(mces)
    return results

print("  ✓ Calibration functions ready.")
print("    Binary  : fit_platt, fit_isotonic + transform")
print("    Multi   : fit_multiclass_isotonic + transform")
print("    Metrics : calc_ece_mce, binary/multiclass_calibration_metrics")

## Cell 4 · Stage 1 Calibration

In [ ]:
# ── Cell 4b · Stage 1 — Uncalibrated is also a candidate ─────────────

stage1_results = {}
stage1_calibrators = {}

for scn in SCENARIOS:
    print(f"\n{'═' * 60}")
    print(f"  STAGE 1 — {scn}")
    print(f"{'═' * 60}")

    oof  = prob_data[f"stage1_{scn}_oof"]
    test = prob_data[f"stage1_{scn}_test"]

    y_oof  = oof["target"].values
    y_test = test["target"].values
    p_oof  = oof["prob_IAS"].values
    p_test = test["prob_IAS"].values

    # ── Baseline (uncalibrated) ───────────────────────────────────────
    bl_test = binary_calibration_metrics(y_test, p_test, label=f"{scn}_TEST_uncal")
    bl_oof  = binary_calibration_metrics(y_oof,  p_oof,  label=f"{scn}_OOF_uncal")
    print(f"\n  Uncalibrated (test) → Brier={bl_test['Brier']:.4f}  ECE={bl_test['ECE']:.4f}  MCE={bl_test['MCE']:.4f}")

    # ── Platt Scaling ─────────────────────────────────────────────────
    platt = fit_platt(y_oof, p_oof)
    p_test_platt = transform_platt(platt, p_test)
    p_oof_platt  = transform_platt(platt, p_oof)
    m_platt = binary_calibration_metrics(y_test, p_test_platt, label=f"{scn}_TEST_platt")
    print(f"  Platt (test)        → Brier={m_platt['Brier']:.4f}  ECE={m_platt['ECE']:.4f}  MCE={m_platt['MCE']:.4f}")

    # ── Isotonic Regression ───────────────────────────────────────────
    iso = fit_isotonic(y_oof, p_oof)
    p_test_iso = transform_isotonic(iso, p_test)
    p_oof_iso  = transform_isotonic(iso, p_oof)
    m_iso = binary_calibration_metrics(y_test, p_test_iso, label=f"{scn}_TEST_isotonic")
    print(f"  Isotonic (test)     → Brier={m_iso['Brier']:.4f}  ECE={m_iso['ECE']:.4f}  MCE={m_iso['MCE']:.4f}")

    # ── Select the best method (by ECE, including uncalibrated) ───────
    candidates = {
        "uncalibrated": (bl_test["ECE"], bl_test["Brier"], None, p_test, p_oof, bl_test),
        "platt":        (m_platt["ECE"], m_platt["Brier"], platt, p_test_platt, p_oof_platt, m_platt),
        "isotonic":     (m_iso["ECE"],   m_iso["Brier"],   iso,   p_test_iso,   p_oof_iso,   m_iso),
    }
    # Primary: ECE, secondary: Brier
    best_method = min(candidates, key=lambda k: (candidates[k][0], candidates[k][1]))
    _, _, best_cal, best_probs_test, best_probs_oof, best_metrics = candidates[best_method]

    if best_method == "uncalibrated":
        print(f"\n  ★ Selected: UNCALIBRATED (already well calibrated)")
    else:
        pct = (1 - best_metrics['ECE'] / bl_test['ECE']) * 100
        print(f"\n  ★ Selected: {best_method.upper()}")
        print(f"    ECE: {bl_test['ECE']:.4f} → {best_metrics['ECE']:.4f} ({pct:+.1f}%)")

    stage1_calibrators[scn] = {
        "method": best_method,
        "calibrator": best_cal  # None if uncalibrated
    }
    stage1_results[scn] = {
        "uncal_oof": bl_oof, "uncal_test": bl_test,
        "platt_test": m_platt, "isotonic_test": m_iso,
        "best_method": best_method,
        "best_test": best_metrics,
        "best_probs_test": best_probs_test,
        "best_probs_oof": best_probs_oof,
    }

print(f"\n{'═' * 60}")
print("  Stage 1 calibration complete.")
print(f"  CBC_Only → {stage1_calibrators['CBC_Only']['method']}")
print(f"  CBC_BIO  → {stage1_calibrators['CBC_BIO']['method']}")

## Cell 5 · Stage 2 Calibration — Fit (OOF) & Evaluate (Test)

In [ ]:
# ── Cell 5 · Stage 2 Calibration — Fit (OOF) & Evaluate (Test) ───────

STAGE2_CLASSES = ["DEA", "HA", "HGB_HTZ", "Normal"]
STAGE2_PROB_COLS = [f"prob_{c}" for c in STAGE2_CLASSES]

stage2_results = {}
stage2_calibrators = {}

for scn in SCENARIOS:
    print(f"\n{'═' * 60}")
    print(f"  STAGE 2 — {scn}")
    print(f"{'═' * 60}")

    oof  = prob_data[f"stage2_{scn}_oof"]
    test = prob_data[f"stage2_{scn}_test"]

    # One-hot encode target
    y_oof_labels  = oof["target"].values
    y_test_labels = test["target"].values
    n_classes = len(STAGE2_CLASSES)

    y_oof_oh  = np.eye(n_classes)[y_oof_labels]
    y_test_oh = np.eye(n_classes)[y_test_labels]

    p_oof  = oof[STAGE2_PROB_COLS].values
    p_test = test[STAGE2_PROB_COLS].values

    # ── Baseline (uncalibrated) ───────────────────────────────────────
    bl_test = multiclass_calibration_metrics(
        y_test_oh, p_test, STAGE2_CLASSES, label=f"{scn}_TEST_uncal")
    bl_oof = multiclass_calibration_metrics(
        y_oof_oh, p_oof, STAGE2_CLASSES, label=f"{scn}_OOF_uncal")

    print(f"\n  Uncalibrated (test):")
    print(f"    Brier_macro={bl_test['Brier_macro']:.4f}  ECE_macro={bl_test['ECE_macro']:.4f}  MCE_macro={bl_test['MCE_macro']:.4f}")
    for cls in STAGE2_CLASSES:
        print(f"      {cls:10s} → Brier={bl_test[f'Brier_{cls}']:.4f}  ECE={bl_test[f'ECE_{cls}']:.4f}")

    # ── Isotonic (per-class OvR) ──────────────────────────────────────
    iso_cals = fit_multiclass_isotonic(y_oof_oh, p_oof, STAGE2_CLASSES)
    p_test_iso = transform_multiclass_isotonic(iso_cals, p_test, STAGE2_CLASSES)
    p_oof_iso  = transform_multiclass_isotonic(iso_cals, p_oof,  STAGE2_CLASSES)

    m_iso = multiclass_calibration_metrics(
        y_test_oh, p_test_iso, STAGE2_CLASSES, label=f"{scn}_TEST_isotonic")

    print(f"\n  Isotonic per-class (test):")
    print(f"    Brier_macro={m_iso['Brier_macro']:.4f}  ECE_macro={m_iso['ECE_macro']:.4f}  MCE_macro={m_iso['MCE_macro']:.4f}")
    for cls in STAGE2_CLASSES:
        print(f"      {cls:10s} → Brier={m_iso[f'Brier_{cls}']:.4f}  ECE={m_iso[f'ECE_{cls}']:.4f}")

    # ── Selection: uncalibrated vs isotonic (by ECE_macro) ────────────
    if m_iso["ECE_macro"] < bl_test["ECE_macro"]:
        best_method = "isotonic"
        best_metrics = m_iso
        best_cal = iso_cals
        best_probs_test = p_test_iso
        best_probs_oof  = p_oof_iso
        pct = (1 - m_iso['ECE_macro'] / bl_test['ECE_macro']) * 100
        print(f"\n  ★ Selected: ISOTONIC")
        print(f"    ECE_macro: {bl_test['ECE_macro']:.4f} → {m_iso['ECE_macro']:.4f} ({pct:+.1f}%)")
    else:
        best_method = "uncalibrated"
        best_metrics = bl_test
        best_cal = None
        best_probs_test = p_test
        best_probs_oof  = p_oof
        print(f"\n  ★ Selected: UNCALIBRATED (already well calibrated)")

    # ── Row sum check ─────────────────────────────────────────────────
    if best_method == "isotonic":
        sums = best_probs_test.sum(axis=1)
        print(f"    Normalization check: min={sums.min():.6f} max={sums.max():.6f}")

    stage2_calibrators[scn] = {"method": best_method, "calibrator": best_cal}
    stage2_results[scn] = {
        "uncal_oof": bl_oof, "uncal_test": bl_test,
        "isotonic_test": m_iso,
        "best_method": best_method,
        "best_test": best_metrics,
        "best_probs_test": best_probs_test,
        "best_probs_oof": best_probs_oof,
    }

print(f"\n{'═' * 60}")
print("  Stage 2 calibration complete.")
print(f"  CBC_Only → {stage2_calibrators['CBC_Only']['method']}")
print(f"  CBC_BIO  → {stage2_calibrators['CBC_BIO']['method']}")

## Cell 6 · Calibration Summary Table

## Cell 7 · Reliability Diagrams — 2×2 Panel

In [ ]:
# ── Cell 7 · Reliability Diagrams — 2×2 Panel ────────────────────────
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

N_BINS = 10

fig, axes = plt.subplots(2, 2, figsize=(7.5, 7), dpi=150)

panel_config = [
    # (row, col, stage, scenario, title)
    (0, 0, 1, "CBC_Only",  "Stage 1 · CBC Only"),
    (0, 1, 1, "CBC_BIO",   "Stage 1 · CBC + Biochemistry"),
    (1, 0, 2, "CBC_Only",  "Stage 2 · CBC Only"),
    (1, 1, 2, "CBC_BIO",   "Stage 2 · CBC + Biochemistry"),
]

for row, col, stg, scn, title in panel_config:
    ax = axes[row, col]

    if stg == 1:
        # ── Binary: prob_IAS vs target ────────────────────────────────
        test = prob_data[f"stage1_{scn}_test"]
        y_true = test["target"].values
        p_uncal = test["prob_IAS"].values

        # Uncalibrated
        frac_pos_u, mean_pred_u = calibration_curve(y_true, p_uncal, n_bins=N_BINS)
        ax.plot(mean_pred_u, frac_pos_u, 's-',
                color=PALETTE["base2"], ms=5, lw=1.2, label="Uncalibrated")

        # Calibrated (if a method was selected)
        r = stage1_results[scn]
        if r["best_method"] != "uncalibrated":
            p_cal = r["best_probs_test"]
            frac_pos_c, mean_pred_c = calibration_curve(y_true, p_cal, n_bins=N_BINS)
            ax.plot(mean_pred_c, frac_pos_c, 'o-',
                    color=PALETTE["highlight"], ms=5, lw=1.2,
                    label=f'{r["best_method"].capitalize()}')

    else:
        # ── Multiclass: macro-average reliability curve ───────────────
        test = prob_data[f"stage2_{scn}_test"]
        y_labels = test["target"].values
        n_classes = len(STAGE2_CLASSES)
        y_oh = np.eye(n_classes)[y_labels]
        p_uncal = test[STAGE2_PROB_COLS].values

        # Per-class average → macro reliability curve
        all_frac_u, all_mean_u = [], []
        for i, cls in enumerate(STAGE2_CLASSES):
            fp, mp = calibration_curve(y_oh[:, i], p_uncal[:, i],
                                        n_bins=N_BINS, strategy="uniform")
            all_frac_u.append(fp)
            all_mean_u.append(mp)

        # Plot each class separately
        _disp = {'DEA': 'IDA', 'HGB_HTZ': 'HGB HTZ'}
        for i, cls in enumerate(STAGE2_CLASSES):
            color = CLASS_COLORS.get(cls, PALETTE["base1"])   # color key stays internal
            ax.plot(all_mean_u[i], all_frac_u[i], 'o-',
                    color=color, ms=3, lw=1, alpha=0.7,
                    label=_disp.get(cls, cls))                 # legend display

        # If calibrated
        r = stage2_results[scn]
        if r["best_method"] != "uncalibrated":
            p_cal = r["best_probs_test"]
            for i, cls in enumerate(STAGE2_CLASSES):
                fp_c, mp_c = calibration_curve(y_oh[:, i], p_cal[:, i],
                                                n_bins=N_BINS, strategy="uniform")
                ax.plot(mp_c, fp_c, 's--', color=CLASS_COLORS.get(cls, PALETTE["base1"]),
                        ms=3, lw=1, alpha=0.5)

    # ── Reference line and formatting ────────────────────────────────────
    ax.plot([0, 1], [0, 1], '--', color='#999999', lw=0.8)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9, fontweight='bold')

    if row == 1:
        ax.set_xlabel("Mean predicted probability", fontsize=8)
    if col == 0:
        ax.set_ylabel("Fraction of positives", fontsize=8)

    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6, loc="lower right", frameon=False)

for ax_row in axes:
    for ax in ax_row:
        despine_all(ax)
fig.suptitle("Reliability Diagrams — Before & After Calibration",
             fontsize=10, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])

# ── Save ────────────────────────────────────────────────────────────
plots_dir = PATHS.module_dir("m1_calibration") / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
save_fig(fig, plots_dir, "reliability_diagrams_2x2")
plt.show()
print(f"  ✓ Saved → {plots_dir}/reliability_diagrams_2x2")

## Cell 8 · ECE Bar Chart — Before vs After

In [ ]:
# ── Cell 8 · ECE Bar Chart — Before vs After ─────────────────────────
# (This block is inserted BEFORE the existing "Cell 8c · Overlap fix" plotting code.)

# ── Collect ECE bar data from stage1_results / stage2_results ─────────────────
# 4 grup: S1 CBC_Only, S1 CBC_BIO, S2 CBC_Only, S2 CBC_BIO
# Display labels (internal keys preserved; x-axis display only):
_panel_specs = [
    ("Stage 1\nCBC Only", stage1_results, "CBC_Only", "ECE"),
    ("Stage 1\nCBC+BIO",  stage1_results, "CBC_BIO",  "ECE"),
    ("Stage 2\nCBC Only", stage2_results, "CBC_Only", "ECE_macro"),
    ("Stage 2\nCBC+BIO",  stage2_results, "CBC_BIO",  "ECE_macro"),
]

labels         = []
ece_before     = []
ece_after      = []
was_calibrated = []

for disp_name, results, scn, ece_key in _panel_specs:
    r = results[scn]
    before = r["uncal_test"][ece_key]
    calibrated = (r["best_method"] != "uncalibrated")
    after = r["best_test"][ece_key] if calibrated else before
    labels.append(disp_name)
    ece_before.append(before)
    ece_after.append(after)
    was_calibrated.append(calibrated)

x = np.arange(len(labels))
w = 0.35

# ── Plotting ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3.5), dpi=150)

for i in range(len(labels)):
    if was_calibrated[i]:
        # Uncalibrated bar (left)
        ax.bar(x[i] - w/2, ece_before[i], w,
               color=PALETTE["base2"], edgecolor="white", lw=0.5, alpha=0.85)
        ax.text(x[i] - w/2, ece_before[i] + 0.002,
                f'{ece_before[i]:.3f}', ha='center', va='bottom', fontsize=6.5)

        # Calibrated bar (right)
        ax.bar(x[i] + w/2, ece_after[i], w,
               color=PALETTE["highlight"], edgecolor="white", lw=0.5)
        ax.text(x[i] + w/2, ece_after[i] + 0.002,
                f'{ece_after[i]:.3f}', ha='center', va='bottom', fontsize=6.5)

        # Improvement percentage
        pct = (1 - ece_after[i] / ece_before[i]) * 100
        ax.annotate(f'{pct:+.0f}%', xy=(x[i], (ece_before[i] + ece_after[i]) / 2),
                    fontsize=6, fontstyle='italic', color=PALETTE["highlight"],
                    ha='center')
    else:
        # Single wide bar — green
        ax.bar(x[i], ece_before[i], w * 1.8,
               color=PALETTE["accent1"], edgecolor="white", lw=0.5, alpha=0.85)
        ax.text(x[i], ece_before[i] + 0.002,
                f'{ece_before[i]:.3f}', ha='center', va='bottom', fontsize=6.5)
        # Place the note in the middle of the bar (not below)
        ax.text(x[i], ece_before[i] / 2, "well-\ncalibrated",
                ha='center', va='center', fontsize=5, fontstyle='italic',
                color='white', fontweight='bold')

# ── Legend ────────────────────────────────────────────────────────────
from matplotlib.patches import Patch
legend_items = [
    Patch(facecolor=PALETTE["base2"], label="Uncalibrated"),
    Patch(facecolor=PALETTE["highlight"], label="After isotonic"),
    Patch(facecolor=PALETTE["accent1"], label="Retained (well-calibrated)"),
]
ax.legend(handles=legend_items, fontsize=6.5, frameon=False, loc="upper right")

ax.set_ylabel("ECE", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=7)
ax.set_ylim(0, max(ece_before) * 1.3)
ax.set_title("Expected Calibration Error", fontsize=9, fontweight='bold')
despine_all(ax)
ax.spines['bottom'].set_visible(True)

plt.tight_layout()
save_fig(fig, plots_dir, "ece_bar_chart")
plt.show()

## Cell 9 · Save Calibrated Probabilities

In [ ]:
# ── Cell 9 · Save calibrated probabilities ───────────────────────────

saved_files = []

for scn in SCENARIOS:
    # ── Stage 1 ───────────────────────────────────────────────────────
    s1 = stage1_calibrators[scn]
    for split in ["oof", "test"]:
        df = prob_data[f"stage1_{scn}_{split}"].copy()

        if s1["method"] == "uncalibrated":
            # Keep probabilities as-is
            df["prob_IAS_cal"] = df["prob_IAS"]
            df["prob_DAS_cal"] = df["prob_DAS"]
        elif s1["method"] == "isotonic":
            df["prob_IAS_cal"] = transform_isotonic(s1["calibrator"], df["prob_IAS"].values)
            df["prob_DAS_cal"] = 1 - df["prob_IAS_cal"]
        elif s1["method"] == "platt":
            df["prob_IAS_cal"] = transform_platt(s1["calibrator"], df["prob_IAS"].values)
            df["prob_DAS_cal"] = 1 - df["prob_IAS_cal"]

        df["calibration_method"] = s1["method"]

        fname = f"stage1_{scn.lower()}_{split}_calibrated.parquet"
        fpath = PATHS.probabilities / fname
        df.to_parquet(fpath, index=False)
        saved_files.append(fname)

    # ── Stage 2 ───────────────────────────────────────────────────────
    s2 = stage2_calibrators[scn]
    for split in ["oof", "test"]:
        df = prob_data[f"stage2_{scn}_{split}"].copy()

        if s2["method"] == "uncalibrated":
            for cls in STAGE2_CLASSES:
                df[f"prob_{cls}_cal"] = df[f"prob_{cls}"]
        else:
            p_raw = df[STAGE2_PROB_COLS].values
            p_cal = transform_multiclass_isotonic(s2["calibrator"], p_raw, STAGE2_CLASSES)
            for i, cls in enumerate(STAGE2_CLASSES):
                df[f"prob_{cls}_cal"] = p_cal[:, i]

        df["calibration_method"] = s2["method"]

        fname = f"stage2_{scn.lower()}_{split}_calibrated.parquet"
        fpath = PATHS.probabilities / fname
        df.to_parquet(fpath, index=False)
        saved_files.append(fname)

print("  ✓ Calibrated parquet files:")
for f in saved_files:
    print(f"    → {f}")
print(f"\n  Total {len(saved_files)} files → {PATHS.probabilities}")

## Cell 10 · Save Calibrator Models

In [ ]:
# ── Cell 10 · Save calibrator models ──────────────────────────────────
import joblib

cal_dir = PATHS.module_dir("m1_calibration") / "calibrators"
cal_dir.mkdir(parents=True, exist_ok=True)

cal_registry = {}

for scn in SCENARIOS:
    # Stage 1
    s1 = stage1_calibrators[scn]
    s1_meta = {
        "stage": 1, "scenario": scn, "method": s1["method"],
        "positive_class": "IAS", "classes": STAGE1_CLASSES,
    }
    if s1["calibrator"] is not None:
        s1_path = cal_dir / f"stage1_{scn.lower()}_calibrator.joblib"
        joblib.dump({"calibrator": s1["calibrator"], "meta": s1_meta}, s1_path)
        print(f"  ✓ {s1_path.name} ({s1['method']})")
    else:
        s1_path = None
        print(f"  ○ Stage1 {scn}: uncalibrated — no calibrator saved")
    cal_registry[f"stage1_{scn}"] = s1_meta

    # Stage 2
    s2 = stage2_calibrators[scn]
    s2_meta = {
        "stage": 2, "scenario": scn, "method": s2["method"],
        "classes": STAGE2_CLASSES,
    }
    if s2["calibrator"] is not None:
        s2_path = cal_dir / f"stage2_{scn.lower()}_calibrator.joblib"
        joblib.dump({"calibrator": s2["calibrator"], "meta": s2_meta}, s2_path)
        print(f"  ✓ {s2_path.name} ({s2['method']})")
    else:
        s2_path = None
        print(f"  ○ Stage2 {scn}: uncalibrated — no calibrator saved")
    cal_registry[f"stage2_{scn}"] = s2_meta

# Save the registry (which model uses which method)
registry_path = cal_dir / "calibration_registry.joblib"
joblib.dump(cal_registry, registry_path)
print(f"\n  ✓ Registry saved → {registry_path.name}")
print(f"  Directory: {cal_dir}")

## Cell 11 · Generate src/m1_calibration.py module

In [ ]:
# ── Cell 11 · Generate src/m1_calibration.py module ───────────────────

m1_module_code = '''"""
m1_calibration.py — Probability Calibration Module
===================================================
Created in Chat 1. Usage in later modules:
    from m1_calibration import load_calibrated_probs, apply_calibration
"""

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss


# ═══════════════════════════════════════════════════════════════════════
# LOADING
# ═══════════════════════════════════════════════════════════════════════

def load_calibrated_probs(paths, stage, scenario, split):
    """Load calibrated probabilities.

    Parameters
    ----------
    paths : CDSPaths
    stage : int (1 or 2)
    scenario : str ("CBC_Only" or "CBC_BIO")
    split : str ("oof" or "test")

    Returns
    -------
    pd.DataFrame
    """
    fname = f"stage{stage}_{scenario.lower()}_{split}_calibrated.parquet"
    return pd.read_parquet(paths.probabilities / fname)


def load_calibrator(paths, stage, scenario):
    """Load the saved calibrator. Returns None if uncalibrated.

    Returns
    -------
    dict with keys 'calibrator' and 'meta', or None
    """
    cal_dir = paths.module_dir("m1_calibration") / "calibrators"
    fpath = cal_dir / f"stage{stage}_{scenario.lower()}_calibrator.joblib"
    if fpath.exists():
        return joblib.load(fpath)
    return None


def load_calibration_registry(paths):
    """Load the calibration registry."""
    cal_dir = paths.module_dir("m1_calibration") / "calibrators"
    return joblib.load(cal_dir / "calibration_registry.joblib")


# ═══════════════════════════════════════════════════════════════════════
# TRANSFORM
# ═══════════════════════════════════════════════════════════════════════

def apply_calibration_binary(calibrator_obj, prob_positive):
    """Apply calibration to a binary probability.

    Parameters
    ----------
    calibrator_obj : dict from load_calibrator, or None
    prob_positive : np.array, positive class probability

    Returns
    -------
    np.array — calibrated probabilities
    """
    if calibrator_obj is None:
        return prob_positive

    method = calibrator_obj["meta"]["method"]
    cal = calibrator_obj["calibrator"]

    if method == "uncalibrated":
        return prob_positive
    elif method == "isotonic":
        return cal.predict(prob_positive)
    elif method == "platt":
        logits = np.log(np.clip(prob_positive, 1e-8, 1-1e-8) /
                        (1 - np.clip(prob_positive, 1e-8, 1-1e-8)))
        return cal.predict_proba(logits.reshape(-1, 1))[:, 1]
    else:
        raise ValueError(f"Unknown calibration method: {method}")


def apply_calibration_multiclass(calibrator_obj, prob_matrix, class_names):
    """Apply calibration to a multiclass probability matrix + normalize.

    Parameters
    ----------
    calibrator_obj : dict from load_calibrator, or None
    prob_matrix : np.array (n_samples, n_classes)
    class_names : list of str

    Returns
    -------
    np.array — calibrated & normalized probabilities
    """
    if calibrator_obj is None:
        return prob_matrix

    method = calibrator_obj["meta"]["method"]

    if method == "uncalibrated":
        return prob_matrix
    elif method == "isotonic":
        cals = calibrator_obj["calibrator"]
        cal_probs = np.zeros_like(prob_matrix)
        for i, cls in enumerate(class_names):
            cal_probs[:, i] = cals[cls].predict(prob_matrix[:, i])
        row_sums = cal_probs.sum(axis=1, keepdims=True)
        row_sums = np.where(row_sums == 0, 1, row_sums)
        return cal_probs / row_sums
    else:
        raise ValueError(f"Unknown calibration method: {method}")


# ═══════════════════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════════════════

def calc_ece_mce(y_true, y_prob, n_bins=10):
    """Expected & Maximum Calibration Error."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece, mce = 0.0, 0.0
    for i, (lo, hi) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
        mask = (y_prob >= lo) & (y_prob <= hi) if i == 0 else (y_prob > lo) & (y_prob <= hi)
        n_bin = mask.sum()
        if n_bin == 0:
            continue
        gap = abs(y_true[mask].mean() - y_prob[mask].mean())
        ece += (n_bin / len(y_true)) * gap
        mce = max(mce, gap)
    return ece, mce
'''

# Save
src_path = PATHS.src / "m1_calibration.py"
with open(src_path, "w", encoding="utf-8") as f:
    f.write(m1_module_code.strip())

print(f"  ✓ Module saved → {src_path}")

# Import test
from m1_calibration import load_calibrated_probs, load_calibration_registry
reg = load_calibration_registry(PATHS)
print(f"  ✓ Import test passed. Registry: {list(reg.keys())}")
for k, v in reg.items():
    print(f"    {k}: {v['method']}")

## Cell 12 · Update manifest — M1